In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **DUMMY MODEL**

In [2]:
# Predicting any 3 options for all questions
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': 'A B C'
})
submission.to_csv('submission.csv', index=False)


# # Predicting the 3 options which have the highest frequency of occurence for all the questions

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # Find top 3 most frequent answers
# top3_answers = train['answer'].value_counts().index[:3].tolist()
# prediction = ' '.join(top3_answers)
# print(f"Top 3 answers: {prediction}")

# submission = pd.DataFrame({
#     'ID': test['id'],
#     'Prediction': prediction
# })
# submission.to_csv('submission.csv', index=False)


# # Predicting first option as mode and any other 2 options as the next 2 for all questions

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # Find the most common answer in training data
# mode_answer = train['answer'].mode()[0]

# # Create all 5 options and put mode first
# options = ['A', 'B', 'C', 'D', 'E']
# options.remove(mode_answer)
# top3 = mode_answer + ' ' + options[0] + ' ' + options[1]

# submission = pd.DataFrame({
#     'ID': test['id'],
#     'Prediction': top3
# })
# submission.to_csv('submission.csv', index=False)
# print(f"Mode answer: {mode_answer}, Prediction: {top3}")

# Understanding the dataset

In [3]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nFirst few rows:")
print(train.head())
print("\nAnswer distribution:")
print(train['answer'].value_counts())
print("\nSample question:")
print(train['prompt'][0])
print(train[['A','B','C','D','E']].iloc[0])

Train shape: (2000, 8)
Test shape: (500, 7)

First few rows:
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegg

Key observations:

* Questions are text-heavy (long sentences as options, not just single words)
* Answer distribution: B is most common (490), E is least (324)
* This is a semantic understanding problem — you need to understand meaning, not just keywords
* Thus, a simple TF-IDF model won't work well because the options are very similar in wording.**

# **Determining how long the texts are, which determines what model architecture and tokenizer settings to use**

In [4]:
# Check average length of text
import pandas as pd
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print("Avg prompt length (words):", train['prompt'].str.split().str.len().mean())
print("Avg option A length (words):", train['A'].str.split().str.len().mean())
print("Max prompt length:", train['prompt'].str.split().str.len().max())

Avg prompt length (words): 18.1465
Avg option A length (words): 26.146
Max prompt length: 51


* The texts are manageable in length (avg ~18 words for prompt, ~26 for options, max 51).
* This means we can use max_length=128 or 256 for tokenization.

# Model 1


* Combine prompt + each option into one string
* Convert to numerical features using a simple embedding layer
* Pass through a feedforward neural network
* Output which option is most likely correct


In [5]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print("Could not load WANDB_API_KEY from Kaggle Secrets:", e)

In [6]:
# ---- Sanity check: train/val leakage ----
# A near-perfect validation accuracy is a red flag, not a milestone -- it usually
# means the val split contains near-duplicates of training rows. This dataset has
# templated question variants (e.g. several differently-worded Heidegger questions
# with very similar option text), so check for leakage before trusting the number.
import pandas as pd
from sklearn.model_selection import train_test_split as _tts_check

_train_check = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
_tr, _val = _tts_check(_train_check, test_size=0.2, random_state=42, stratify=_train_check['answer'])

def _normalize(text):
    return str(text).lower().strip()

_tr_norm = set(_tr['prompt'].apply(_normalize))
_val_norm = set(_val['prompt'].apply(_normalize))
_overlap = _tr_norm & _val_norm
print(f"Exact prompt overlap between train/val: {len(_overlap)} rows")

_tr_sig = set(zip(_tr['answer'], _tr['A'], _tr['B'], _tr['C'], _tr['D'], _tr['E']))
_val_sig = set(zip(_val['answer'], _val['A'], _val['B'], _val['C'], _val['D'], _val['E']))
_sig_overlap = _tr_sig & _val_sig
print(f"Rows with identical option sets in both train/val: {len(_sig_overlap)}")


Exact prompt overlap between train/val: 62 rows
Rows with identical option sets in both train/val: 153


In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb

# ---- Load Data ----
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# ---- Build Vocabulary ----
def tokenize(text):
    return str(text).lower().split()

all_text = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train[col].fillna('').apply(lambda x: all_text.extend(tokenize(x)))
    test[col].fillna('').apply(lambda x: all_text.extend(tokenize(x)))

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in Counter(all_text).items():
    if count >= 2:
        vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")

import json
with open('vocab.json', 'w') as f:
   json.dump(vocab, f)

# ---- Encode Text ----
def encode(text, max_len=64):
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids += [0] * (max_len - len(ids))
    return ids

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
options = ['A', 'B', 'C', 'D', 'E']

# ---- Dataset ----
class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = encode(row['prompt'])
        opts = [encode(row[opt]) for opt in options]
        # Combine prompt with each option
        combined = [prompt + opt for opt in opts]
        x = torch.tensor(combined, dtype=torch.long)  # shape: (5, 128)
        if not self.is_test:
            y = torch.tensor(label_map[row['answer']], dtype=torch.long)
            return x, y
        return x

# ---- Model ----
class MCQModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, max_len=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),  # fixed: embed_dim not embed_dim*128
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x: (batch, 5, 128)
        x = self.embedding(x)           # (batch, 5, 128, embed_dim)
        x = x.mean(dim=2)               # (batch, 5, embed_dim)
        logits = self.fc(x).squeeze(-1) # (batch, 5)
        return logits


def map_at_3(y_true_idx, logits):
    """MAP@3 over a batch of predictions -- the competition's actual metric,
    reported here alongside accuracy/F1 for the final report."""
    top3 = torch.topk(logits, 3, dim=1).indices
    scores = []
    for true_idx, pred_idx in zip(y_true_idx, top3):
        pred_list = pred_idx.tolist()
        scores.append(1.0 / (pred_list.index(true_idx) + 1) if true_idx in pred_list else 0.0)
    return sum(scores) / len(scores)

# ---- Training ----
wandb.init(project="24f3002284-t22026", name="model1-scratch", config={
    "embed_dim": 64,
    "hidden_dim": 128,
    "epochs": 10,
    "batch_size": 32,
    "lr": 1e-3
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Stratified split preserves the class balance in both train and val, and
# matches the splitting strategy used for Models 2 and 3 -- so the three
# wandb runs are actually comparable, not just superficially similar.
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=train['answer'])

train_loader = DataLoader(MCQDataset(train_df), batch_size=32, shuffle=True)
val_loader = DataLoader(MCQDataset(val_df), batch_size=32)

model = MCQModel(len(vocab)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

best_val_f1 = -1
for epoch in range(10):
    model.train()
    total_loss = 0.0
    train_true, train_pred = [], []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        train_true += y.cpu().tolist()
        train_pred += logits.argmax(1).cpu().tolist()

    train_acc = accuracy_score(train_true, train_pred)
    train_f1 = f1_score(train_true, train_pred, average='macro')

    # Validation
    model.eval()
    val_true, val_pred, val_logits_all = [], [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            val_true += y.cpu().tolist()
            val_pred += logits.argmax(1).cpu().tolist()
            val_logits_all.append(logits.cpu())

    val_acc = accuracy_score(val_true, val_pred)
    val_f1 = f1_score(val_true, val_pred, average='macro')
    val_map3 = map_at_3(val_true, torch.cat(val_logits_all, dim=0))

    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, "
          f"Train Acc={train_acc:.4f}, Train F1={train_f1:.4f}, "
          f"Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}, Val MAP@3={val_map3:.4f}")

    wandb.log({
        "epoch": epoch + 1, "loss": total_loss / len(train_loader),
        "train_acc": train_acc, "train_f1": train_f1,
        "val_acc": val_acc, "val_f1": val_f1, "val_map3": val_map3,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'model1_scratch_best.pth')
        wandb.save('model1_scratch_best.pth')

wandb.summary["best_val_f1"] = best_val_f1
wandb.finish()
print("Training done!")


Vocab size: 3816


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Using device: cpu


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 1: Loss=1.6056, Train Acc=0.2456, Train F1=0.2451, Val Acc=0.4525, Val F1=0.4507, Val MAP@3=0.6204
Epoch 2: Loss=1.5888, Train Acc=0.3675, Train F1=0.3646, Val Acc=0.5500, Val F1=0.5445, Val MAP@3=0.6946
Epoch 3: Loss=1.5119, Train Acc=0.5031, Train F1=0.5008, Val Acc=0.6625, Val F1=0.6541, Val MAP@3=0.7696
Epoch 4: Loss=1.2652, Train Acc=0.6319, Train F1=0.6247, Val Acc=0.7525, Val F1=0.7381, Val MAP@3=0.8438
Epoch 5: Loss=0.9052, Train Acc=0.7425, Train F1=0.7361, Val Acc=0.7950, Val F1=0.7822, Val MAP@3=0.8738
Epoch 6: Loss=0.6390, Train Acc=0.8187, Train F1=0.8101, Val Acc=0.8350, Val F1=0.8228, Val MAP@3=0.9062
Epoch 7: Loss=0.4715, Train Acc=0.8719, Train F1=0.8650, Val Acc=0.8575, Val F1=0.8501, Val MAP@3=0.9208
Epoch 8: Loss=0.3733, Train Acc=0.8944, Train F1=0.8886, Val Acc=0.9175, Val F1=0.9105, Val MAP@3=0.9550
Epoch 9: Loss=0.2929, Train Acc=0.9250, Train F1=0.9205, Val Acc=0.9300, Val F1=0.9259, Val MAP@3=0.9633
Epoch 10: Loss=0.2297, Train Acc=0.9413, Train F1=0.938

epoch,▁▂▃▃▄▅▆▆▇█
loss,███▆▄▃▂▂▁▁
train_acc,▁▂▄▅▆▇▇███
train_f1,▁▂▄▅▆▇▇▇██
val_acc,▁▂▄▅▆▆▇███
val_f1,▁▂▄▅▆▆▇███
val_map3,▁▂▄▅▆▇▇███
best_val_f1,0.94176
epoch,10
loss,0.22965
train_acc,0.94125


Training done!


* This high accuracy might be due to the model memorizing patterns.

**Inference**

In [8]:
# ---- Inference with Model 1 (scratch) ----
# Loads the best checkpoint (by val F1, not just whatever's left from the last
# epoch) and saves raw logits alongside the submission so ensemble.py can later
# combine this with Model 2 and Model 3's predictions.
model.load_state_dict(torch.load('model1_scratch_best.pth', map_location=device))
model.eval()
test_dataset = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=32)

all_logits = []
with torch.no_grad():
    for x in test_loader:
        x = x.to(device)
        logits = model(x)
        all_logits.append(logits.cpu())
all_logits = torch.cat(all_logits, dim=0)
torch.save(all_logits, 'scratch_logits.pt')  # consumed by ensemble.py

top3 = torch.topk(all_logits, 3, dim=1).indices.numpy()
predictions = [' '.join([options[i] for i in pred]) for pred in top3]
submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())

   ID Prediction
0   1      E A D
1   2      B A E
2   3      B D E
3   4      E C A
4   5      C A D


* THis was a simple embedding model. It won't generalize well to unseen test data even if training accuracy is high.
* The 93% training accuracy implies overfitting.

# Milestone 1

In [9]:
# import pandas as pd
# df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv") 

1.Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [10]:
# print("Frequency distribution: ",df["answer"].value_counts())
# print("Max+min freq. sum:",df["answer"].value_counts().max()+df["answer"].value_counts().min())

2.After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [11]:
# import string

# all_words=set()
# for p in df["prompt"]:
#     clean_p=str(p).lower()
#     clean_p=clean_p.translate(str.maketrans("","",string.punctuation))
#     all_words.update(clean_p.split())

# print(len(all_words))

3.Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [12]:
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# row1_prompt=df.iloc[0]['prompt']
# row1_prompt=str(row1_prompt).lower().translate(str.maketrans("","",string.punctuation)).split()

# row1_filtered=[w for w in row1_prompt if w not in ENGLISH_STOP_WORDS]
# len(row1_filtered)

4.Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [13]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# combined_text = df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']
# vectorizer = TfidfVectorizer(stop_words='english')
# tfidf_matrix = vectorizer.fit_transform(combined_text)
# len(vectorizer.get_feature_names_out())

5.Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [14]:
# from sklearn.metrics.pairwise import cosine_similarity

# prompt_vec = vectorizer.transform([df.iloc[0]['prompt']])
# option_a_vec = vectorizer.transform([df.iloc[0]['A']])
# round(cosine_similarity(prompt_vec, option_a_vec)[0][0], 4)

6.Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [15]:
# def get_best_option(row):
#     options = ['A', 'B', 'C', 'D', 'E']
#     prompt_v = vectorizer.transform([row['prompt']])
#     scores = {opt: cosine_similarity(prompt_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
#     return max(scores, key=scores.get)

# matches = df.apply(lambda row: get_best_option(row) == row['answer'], axis=1)
# matches.mean() * 100

7.If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [16]:
# def map_at_3(truth, predictions):
#     if truth in predictions[:3]:
#         rank = predictions.index(truth) + 1
#         return 1 / rank
#     return 0

# map_at_3('C', ['C', 'A', 'B'])

8.If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [17]:
# map_at_3('B', ['D', 'B', 'E'])

9.The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [18]:
# top_answers = df['answer'].value_counts().index.tolist()[:3]
# scores = df['answer'].apply(lambda truth: map_at_3(truth, top_answers))
# scores.mean()

10.The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [19]:
# def get_top3_preds(row):
#     options = ['A', 'B', 'C', 'D', 'E']
#     p_v = vectorizer.transform([row['prompt']])
#     s = {opt: cosine_similarity(p_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
#     return sorted(s, key=s.get, reverse=True)

# all_scores = df.apply(lambda row: map_at_3(row['answer'], get_top3_preds(row)), axis=1)
# all_scores.mean()

# Milestone 2

In [20]:
# import pandas as pd
# import torch
# import numpy as np
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModel, pipeline
# from sentence_transformers import SentenceTransformer, util
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
 
# DATA_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# df = pd.read_csv(DATA_PATH)
# options = ["A", "B", "C", "D", "E"]

In [21]:
# ds = load_dataset("csv", data_files=DATA_PATH)["train"]
 
# def make_combined(example):
#     example["combined_text"] = example["prompt"] + " " + example["A"]
#     return example
 
# ds = ds.map(make_combined)
 
# q1_answer = len(ds[51]["combined_text"])
# print("\nQ1) combined_text char length at row index 51:", q1_answer)

In [22]:
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# q2_answer = tokenizer.vocab_size
# print("Q2) bert-base-uncased vocab size:", q2_answer)

In [23]:
# q3_answer = tokenizer.sep_token_id
# print("Q3) [SEP] token id:", q3_answer)

In [24]:
# all_prompts = [str(p) for p in ds["prompt"]]  # guard against stray non-str/NaN values
# encoded = tokenizer(
#     all_prompts,
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt",
# )
# q4_answer = tuple(encoded["input_ids"].shape)
# print("Q4) input_ids tensor shape:", q4_answer)

In [25]:
# q5_answer = 768 // 12
# print("\nQ5) Per-head attention dimensionality:", q5_answer)

In [26]:
# model = AutoModel.from_pretrained("bert-base-uncased")
# model.eval()
 
# row0_prompt = ds[0]["prompt"]
# inputs_row0 = tokenizer(row0_prompt, return_tensors="pt")  # default: no padding/truncation
 
# with torch.no_grad():
#     out_row0 = model(**inputs_row0)
 
# q6_answer = tuple(out_row0.last_hidden_state.shape)
# print("Q6) last_hidden_state shape (row ID 0 prompt):", q6_answer)

In [27]:
# cls_vec = out_row0.last_hidden_state[0, 0]  # [CLS] is token index 0
# q7_answer = round(cls_vec[:5].sum().item(), 4)
# print("Q7) Sum of first 5 float values of [CLS] embedding:", q7_answer)

In [28]:
# model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
# model_attn.eval()
 
# text_q8 = "Light-ion fusion is a technique."
# inputs_q8 = tokenizer(text_q8, return_tensors="pt")
# tokens_q8 = tokenizer.convert_ids_to_tokens(inputs_q8["input_ids"][0])
# fusion_idx = tokens_q8.index("fusion")
 
# with torch.no_grad():
#     out_q8 = model_attn(**inputs_q8)
 
# last_layer_attn = out_q8.attentions[-1]          # shape: [batch, heads, seq, seq]
# cls_to_fusion = last_layer_attn[0, 0, 0, fusion_idx].item()   # head 0, CLS(row 0) -> fusion(col)
# q8_answer = round(cls_to_fusion, 4)
# print("Q8) Attention weight [CLS] -> 'fusion' (last layer, head 0):", q8_answer,
#       "  (tokens were:", tokens_q8, ")")

In [29]:
# minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
 
# row0 = ds[0]
# emb_prompt_0 = minilm.encode(row0["prompt"], convert_to_tensor=True)
# emb_b_0 = minilm.encode(row0["B"], convert_to_tensor=True)
# q9_answer = round(util.cos_sim(emb_prompt_0, emb_b_0).item(), 4)
# print("\nQ9) Cosine similarity (prompt vs Option B, row ID 0):", q9_answer)

In [30]:
# df = pd.read_csv(DATA_PATH)  # easier to iterate row-wise for this part
 
# def apk3(actual, predicted):
#     if actual in predicted:
#         return 1.0 / (predicted.index(actual) + 1)
#     return 0.0
 
# # Pipeline 1: TF-IDF cosine similarity (per-row TF-IDF fit on prompt + 5 options)
# # NOTE: replace this function with your actual Milestone 1 implementation
# # if it differs (e.g. if you fit TF-IDF globally across the whole corpus
# # instead of per-row), so this stays consistent with your earlier milestone.
# def top3_tfidf(row):
#     docs = [row["prompt"]] + [row[o] for o in options]
#     vec = TfidfVectorizer().fit_transform(docs)
#     sims = cosine_similarity(vec[0:1], vec[1:])[0]
#     ranked = [options[i] for i in np.argsort(-sims)]
#     return ranked[:3]
 
# # Pipeline 2: all-MiniLM-L6-v2 embeddings
# def top3_minilm(row):
#     prompt_emb = minilm.encode(row["prompt"], convert_to_tensor=True)
#     opt_embs = minilm.encode([row[o] for o in options], convert_to_tensor=True)
#     sims = util.cos_sim(prompt_emb, opt_embs)[0].cpu().numpy()
#     ranked = [options[i] for i in np.argsort(-sims)]
#     return ranked[:3]
 
# tfidf_top3s, minilm_top3s = [], []
# aps_tfidf, aps_minilm = [], []
 
# for _, row in df.iterrows():
#     t3_tfidf = top3_tfidf(row)
#     t3_minilm = top3_minilm(row)
#     tfidf_top3s.append(t3_tfidf)
#     minilm_top3s.append(t3_minilm)
#     aps_tfidf.append(apk3(row["answer"], t3_tfidf))
#     aps_minilm.append(apk3(row["answer"], t3_minilm))
 
# mapk3_tfidf = round(np.mean(aps_tfidf), 4)
# mapk3_minilm = round(np.mean(aps_minilm), 4)
# print("Q10a) TF-IDF pipeline MAP@3 (for reference):", mapk3_tfidf)
# print("Q10a) all-MiniLM-L6-v2 pipeline MAP@3:", mapk3_minilm)
 
# q10b_count = sum(
#     1 for i in range(len(df))
#     if df.iloc[i]["answer"] not in tfidf_top3s[i] and df.iloc[i]["answer"] in minilm_top3s[i]
# )
# print("Q10b) Count where TF-IDF misses but MiniLM top-3 hits:", q10b_count)

In [31]:
# zsc = pipeline("zero-shot-classification")  # defaults to facebook/bart-large-mnli
 
# row1 = df.iloc[1]
# candidate_labels = [row1["A"], row1["B"], row1["C"]]
 
# result_softmax = zsc(row1["prompt"], candidate_labels)
# q11_answer = round(result_softmax["scores"][0], 4)
# print("\nQ11) Top-ranked probability (softmax, single-label):", q11_answer)
# print("     (label order returned:", result_softmax["labels"], ")")

In [32]:
# result_sigmoid = zsc(row1["prompt"], candidate_labels, multi_label=True)
# q12_answer = round(abs(sum(result_softmax["scores"]) - sum(result_sigmoid["scores"])), 4)
# print("Q12) |sum(softmax scores) - sum(sigmoid scores)|:", q12_answer)

In [33]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# _t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
# _t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# def gen(prompt_str, max_new_tokens=50):
#     inputs = _t5_tokenizer(prompt_str, return_tensors="pt")
#     output_ids = _t5_model.generate(**inputs, max_new_tokens=max_new_tokens)
#     text = _t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)
#     return [{"generated_text": text}]
 
# row0_df = df.iloc[0]
# prompt_str = (
#     f"Question: {row0_df['prompt']}. Is the correct answer A: {row0_df['A']} or "
#     f"B: {row0_df['B']}? Answer with just the letter A or B."
# )
# gen_output = gen(prompt_str, max_new_tokens=5)
# q13_answer = gen_output[0]["generated_text"]
# print("\nQ13) flan-t5-small exact output string:", repr(q13_answer))